# A — Data Cleaning & Integration Pipeline

*Deliverable A: A1 through A8.*

Runs `src/cleaning.py` which cleans all 3 raw tables, joins them, engineers features,
and writes `master_train.csv` and `master_test.csv` to `data/processed/`.
See `reports/A_cleaning_and_integration.md` for the full written report.

In [ ]:
import subprocess, sys, os
# Run from notebooks/ — adjust cwd to project root
cwd = os.path.abspath('..')
result = subprocess.run(
    [sys.executable, 'src/cleaning.py'],
    capture_output=True, text=True, cwd=cwd
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# A7 — Verify master tables
import pandas as pd
train = pd.read_csv('../data/processed/master_train.csv', parse_dates=['pickup_hour'])
test  = pd.read_csv('../data/processed/master_test.csv',  parse_dates=['pickup_hour'])
print('master_train.csv:', train.shape)
print('master_test.csv: ', test.shape)
print('Zones:', sorted(train.zone.unique()))
print('Train period:', train.pickup_hour.min().date(), 'to', train.pickup_hour.max().date())
print('Test period: ', test.pickup_hour.min().date(),  'to', test.pickup_hour.max().date())
print('Features:', list(train.columns))

In [ ]:
# A1 — Cleaning summary
issues = [
    ('ride_demand_train', 'zone',              'Inconsistent zone spellings',          '50+ spellings', '12 canonical zone names'),
    ('ride_demand_train', 'trips',             'Negative -1 sentinel values',          '681 (0.8%)',    'Median imputation (zone x dow x hour)'),
    ('ride_demand_train', 'trips',             'Outliers > 190 (99.9th pct)',          '85 (0.1%)',     'Capped at 190'),
    ('ride_demand_train', 'trips',             'Missing trip counts',                  '845 (0.99%)',   'Dropped (no label)'),
    ('ride_demand_train', 'zone+pickup_hour',  'Duplicate zone-hour pairs',            '838 (0.99%)',   'Round-mean aggregated'),
    ('weather_hourly',    'timestamp',         'UTC timestamps need +3h conversion',   'all rows',      'tz_convert to Africa/Addis_Ababa'),
    ('weather_hourly',    'temp_c',            'Fahrenheit readings (>50 C)',          '350 (4.69%)',   '(temp-32)*5/9'),
    ('weather_hourly',    'humidity_pct',      'Missing humidity (1-2h gaps)',          '151 (2.02%)',   'ffill'),
    ('weather_hourly',    'rain_mm',           '-9999 sentinel values',               'varies',        'Replace then ffill'),
    ('events_calendar',   'event_type',        'Inconsistent casing/spacing',          '20 rows',       'Lowercase + map to 8 canonical types'),
    ('events_calendar',   'end_datetime',      'Missing end times',                    '4 events',      'Category median duration'),
    ('events_calendar',   'start/end_datetime','End before start',                     '1 event',       'Swapped'),
]
df_log = pd.DataFrame(issues, columns=['file','column','issue','rows_affected','fix'])
print(df_log.to_string(index=False))